# Agent Architectures & LangGraph Fundamentals

This jupyter notebook contains the following:

**Lab A : Agentic patterns & how to choose between them** (Parts A1–A4)

**Lab B : Document approval workflow** (Parts B1–B8)

There is one major concept across these two labs:

> In production, **you do not need to deploy an agent where just a well-defined workflow would have worked** and inherit unnecessary non-determinism, cost, and untestability.


| | Part | You build |
|---|---|---|
| **A** | 1 · The autonomy spectrum | A printed ladder from single call → multi-agent, plus the two selection gates |
| | 2 · One graph per pattern | Five minimal LangGraphs — single call, prompt chain, router, single agent, multi-agent — each run against a **real model** on a real business case, and visualised |
| | 3 · The named agentic patterns | Runnable graphs for **ReAct**, **Planner-Executor**, **Reflection** and **Supervisor-Worker** |
| | 4 · The decision tree | One tree that picks the pattern — and, if multi-agent, the topology — applied to six business problems you justify in writing (solutions included) |
| **B** | 1 · State | A typed state schema with one reducer |
| | 2 · Nodes | Deterministic check + revise functions, plus one LLM-backed revise node |
| | 3 · Wiring | Conditional routing, plus per-node state inspection |
| | 4 · Loops | A revision loop with a guard |
| | 5 · Persistence | Checkpointer + `thread_id`, state history |
| | 6 · Interrupt | `interrupt()` + `Command(resume=...)` |
| | 7 · Durability | SQLite checkpointer, real kernel restart ← **Milestone 5** |
| | 8 · The trap | Double-executed side effects |

**Learning objectives**
1. Justify a pattern choice (single vs. multi-agent, and which multi-agent pattern) for a given problem.
2. Build and checkpoint a stateful LangGraph workflow, including a human-in-the-loop interrupt.

**About the model calls.** Lab A drives every graph with a **real LLM** through `ChatLiteLLM`
(LiteLLM, the program-standard client), so you see what an agent actually does when
a model is in the loop. Set `OPENAI_API_KEY` in your `.env` to run them for real. If no working key is
found, the setup cell detects it once and every model-backed cell falls back to a clearly-labelled
deterministic stub — **the notebook still runs end-to-end and every self-check still passes.**
Lab B's core path is deliberately deterministic: a checkpoint/interrupt lesson must be reproducible.

**How to use this notebook.** Read the markdown before each cell, then run the cell and compare
against the `EXPECTED OUTPUT` docstring at its bottom — that is how you know you are on track. For
model-backed cells the expected output describes the *structure* (which keys, how many loops, what
must be true), not the exact wording, because the wording changes every run.


### Steps to Obtain API Keys

**1. Google (Gemini AI Studio) API Key**

  * Go to [Google AI Studio](https://aistudio.google.com/).
  * Sign in with your Google account.
  * On the left-hand navigation menu, click on **API keys**.
  * Click the **Create API key** button present at the top right corner.
  * Select an existing Google Cloud project or create a new one, then generate and copy your `GOOGLE_API_KEY`.

---
## Setup

In [ ]:
# Pinned for this cohort - do not un-pin, so a framework update can't break the lab mid-session.
# langgraph                     : the graph runtime
# langgraph-checkpoint-sqlite   : the DURABLE checkpointer - a SEPARATE package, required by Part B7
# langchain-litellm             : ChatLiteLLM - LiteLLM (the program-standard client)
#                                 wrapped as a LangChain chat model, so it plugs straight into
#                                 LangGraph: .bind_tools(), message objects, add_messages reducer.
# litellm / python-dotenv       : the underlying client + the .env loader
# (graph pictures need no extra package - .draw_mermaid_png() ships with langgraph)
# (No -q: if an install fails you want the full pip error visible, not silenced.)
%pip install "langgraph>=1.2,<2.0" "langgraph-checkpoint-sqlite>=3.0,<4.0" "python-dotenv>=1.0"
%pip install "litellm>=1.93,<2.0" "langchain-litellm>=0.7,<0.8"

print("Dependencies installed. If pip asks you to restart the kernel, do so, then continue.")

"""
EXPECTED OUTPUT
---------------
(pip install log)
Dependencies installed. If pip asks you to restart the kernel, do so, then continue.
"""

In [ ]:
# Readiness check - confirm every import BEFORE you build anything on it.
from dotenv import load_dotenv
load_dotenv(".env", override=True)  # reads OPENAI_API_KEY etc. from a .env copied from .env.template

import os
checks = {}

def probe(label, fn):
    try:
        fn()
        checks[label] = "ok"
    except Exception as e:
        checks[label] = f"FAILED: {type(e).__name__}: {e}"

probe("langgraph core", lambda: __import__("langgraph.graph", fromlist=["StateGraph"]).StateGraph)
probe("in-memory checkpointer", lambda: __import__("langgraph.checkpoint.memory", fromlist=["InMemorySaver"]).InMemorySaver)
probe("sqlite checkpointer", lambda: __import__("langgraph.checkpoint.sqlite", fromlist=["SqliteSaver"]).SqliteSaver)
probe("interrupt / Command", lambda: __import__("langgraph.types", fromlist=["interrupt"]).interrupt)
probe("prebuilt ToolNode", lambda: __import__("langgraph.prebuilt", fromlist=["ToolNode"]).ToolNode)
probe("ChatLiteLLM", lambda: __import__("langchain_litellm", fromlist=["ChatLiteLLM"]).ChatLiteLLM)

print("Environment readiness")
print("---------------------")
for k, v in checks.items():
    print(f"  {k:>22} : {v}")

if all(v == "ok" for v in checks.values()):
    print("\nCore is green. Run the next cell to connect a model, then start Lab A.")
else:
    print("\nFix the FAILED lines before continuing.")

"""
EXPECTED OUTPUT
---------------
Environment readiness
---------------------
          langgraph core : ok
   in-memory checkpointer : ok
      sqlite checkpointer : ok
      interrupt / Command : ok
        prebuilt ToolNode : ok
             ChatLiteLLM : ok

Core is green. Run the next cell to connect a model, then start Lab A.
"""

### Connecting the model — one handle used by the whole notebook

`ChatLiteLLM` is [LiteLLM](https://docs.litellm.ai/) — this program's standard LLM client — presented
as a LangChain chat model. That matters here for one reason: LangGraph nodes are happiest talking to
something that speaks *messages* and supports `.bind_tools()`, which is exactly what a chat model
gives you. Change one string (`LLM_MODEL`) and the same code runs against OpenAI, Anthropic, Gemini,
Bedrock or a local Ollama model — that portability is the whole point of routing through LiteLLM.

The cell below builds that handle and then **actually calls the model once** to check the key works.
A key that is present but invalid is the most common Day-1 failure, and a variable named
`LLM_ENABLED` that only checks `os.getenv(...)` would happily lie to you about it.


In [ ]:
GEMINI_API_KEY = "paste-your-free-key-here"  # Paste your free key here

In [ ]:
# The single model handle for this notebook. Every model-backed cell below uses ask()/chat_model.
import os
from langchain_litellm import ChatLiteLLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# Cheapest tier that reliably does these tasks. Swap the string to switch provider:
LLM_MODEL = "gemini/gemini-3.1-flash-lite" # Change model here

# temperature=0 => as close to reproducible as a real model gets. Every graph in this notebook is
# meant to be re-run and compared, so we never want creative variation in the control path.
chat_model = ChatLiteLLM(model=LLM_MODEL, temperature=0, api_key=os.getenv("GEMINI_API_KEY", GEMINI_API_KEY))

def _probe_model() -> bool:
    """One real 1-token call. A present-but-invalid key fails HERE, once, with a clear message -
    instead of failing inside a graph node ten cells later where it looks like a LangGraph bug."""
    try:
        chat_model.invoke([HumanMessage(content="Reply with the single word: ok")])
        return True
    except Exception as e:
        print(f"  model unavailable ({type(e).__name__}: {str(e)[:120]})")
        return False

LLM_ENABLED = _probe_model()

def ask(system: str, user: str, model=None) -> str | None:
    """Every plain text->text model call in this notebook goes through here.

    Returns the model's text, or None if no model is available - so each caller can write
        value = ask(...) or <deterministic fallback>
    and stay runnable for a learner whose key is missing or rate-limited."""
    if not LLM_ENABLED:
        return None
    try:
        msgs = [SystemMessage(content=system), HumanMessage(content=user)]
        return (model or chat_model).invoke(msgs).content.strip()
    except Exception as e:                 # transient 429/timeout: degrade, do not crash the lab
        print(f"  [model call failed: {type(e).__name__}] falling back")
        return None

print("LLM_ENABLED :", LLM_ENABLED, "|", LLM_MODEL if LLM_ENABLED else "using deterministic fallbacks")
print("Fallback mode is fully supported: every cell runs and every self-check passes either way.")

---
---
# LAB A · Agentic Patterns

*Here we develop a decision framework to decide on the right design for any problem*.

1. First, we go through agent autonomy; Do we even need a multi-agent setup?
2. Second, we build one minimal LangGraph graph for each pattern on the autonomy spectrum, then the
   four named patterns you will meet in every agent codebase — **ReAct**, **Planner-Executor**,
   **Reflection**, **Supervisor-Worker**.
3. Finally, we put a business problem in front of the decision tree and justify the answer.
   Discussion-oriented: the reasoning matters more than the code output here.

This lab makes the pattern spectrum **concrete**. You will run the smallest working example of each
one — against a real model — on a real business problem, and *visualize* the graph itself.


---
## Part A1 · The autonomy spectrum

Whenever we pick up a problem we are considering to solve with an agentic pipeline, we need to ask a question: **how much of the control flow is decided at design time, and how much at run time?**

```
DECIDED AT DESIGN TIME  ──────────────────────────────────►  DECIDED AT RUN TIME

[1] Single LLM call    [2] Prompt chain   [3] Router      [4] Tool-calling agent  [5] Multi-agent
    no branching       fixed steps        LLM picks       LLM chooses tools       agents delegate
                                          1 of N          and when to stop        to agents

  predictable · easier to test  ─────────────────────►  flexible · possibly costlier · harder to test
```

As per [Anthropic's definitions](https://www.anthropic.com/engineering/building-effective-agents):

Positions **1–3 are basic workflows**: Workflows are systems where LLMs and tools are orchestrated through predefined code paths. A human writes the graph; the LLM fills in content and at most picks a branch.

Positions **4–5 are agents**: Agents are systems where LLMs dynamically direct their own processes and tool usage, maintaining control over how they accomplish tasks

**The rule to carry out of this lab:** push autonomy to the model *only* where the space of valid action sequences is too large or too data-dependent to enumerate. Everywhere else, enumerate it.


Below is a table describing each pattern in the above spectrum:

| # | PATTERN | CLASS | CONTROL FLOW | EXAMPLE |
|---|---|---|---|---|
| 1 | `single_llm_call` | workflow | One call, no branching | Classification, extraction, rewriting |
| 2 | `prompt_chain` | workflow | Fixed sequence of steps, human-defined | Extract -> validate -> format |
| 3 | `router` | workflow | LLM picks one of N enumerated paths | Routing ticket to the correct team |
| 4 | `single_agent` | agent | LLM chooses tools and when to stop | Research a supplier and write a memo |
| 5 | `multi_agent` | agent | Agents delegate to other agents | Legal + finance + compliance contract review |

The four *named* patterns everyone talks about — `ReAct`, `Planner-Executor`, `Reflection`,
`Supervisor-Worker` — are specialisations of rows 4 and 5. We build each one in Part A3.


---
## Part A2 · One minimal graph per pattern

Below are **minimal LangGraph implementations** of each pattern from Part A1, each on a real business
problem, each driven by a **real model call**. Run each cell, read the printed graph, and observe how
much of the control flow moved from your code into the model's hands as you go down the list.

As you run them, keep two questions in mind — Part A4 turns them into a decision tree:

1. **Does this need an agent at all?** Only if the steps cannot be enumerated at design time.
2. **If so, does it need more than one?** Only for distinct expertise, parallel speed-up, or a generator/critic split.

Every cell degrades to a deterministic stub if no model is available, so the graph structure — the
thing you are here to learn — is always visible.


In [ ]:
# Provided - the program-standard graph visualiser (read, run, no changes needed).

from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display, Markdown

def show_graph(app, title: str = ""):
    if title:
        print(title)
        print("-" * len(title))
    try:
        display(Image(app.get_graph().draw_mermaid_png()))      # real PNG, inline
    except Exception as e:
        print(f"(mermaid.ink unavailable - {type(e).__name__}; Mermaid source below)")
        display(Markdown("```mermaid\n" + app.get_graph().draw_mermaid() + "\n```"))

print("show_graph() ready.")


### Pattern 1 · Single LLM call
*Use case: label a support email's urgency.* No branch, no loop. This is the right tool for the large share of work that is simply **input → output**: classification, extraction, rewriting.


In [ ]:
# Pattern 1 - SINGLE LLM CALL.  Use case: label a support email's urgency.
# Control flow decided at DESIGN time: there is none. START -> classify -> END.
class MailState(TypedDict):
    email: str
    urgency: str

VALID_URGENCY = {"high", "normal"}

def classify_urgency(state: MailState) -> dict:
    label = ask(
        system="You label support emails by urgency. Reply with exactly one word: high or normal.",
        user=state["email"],
    )
    # A model returns free text - "High.", " normal\n", "I would say high". NEVER let that reach
    # state raw. Squeeze it back into your typed vocabulary at the boundary, and keep a
    # deterministic fallback for when it does not fit (or when no model is available at all).
    label = (label or "").lower().strip(" .!\n")
    if label not in VALID_URGENCY:
        label = "high" if any(w in state["email"].lower()
                              for w in ("down", "urgent", "cannot", "outage")) else "normal"
    return {"urgency": label}

g = StateGraph(MailState)
g.add_node("classify", classify_urgency)
g.add_edge(START, "classify")
g.add_edge("classify", END)
single_call = g.compile()

print(single_call.invoke({"email": "Production is DOWN, customers cannot check out!", "urgency": ""}))
show_graph(single_call, "Pattern 1 · single LLM call")

"""
EXPECTED OUTPUT
---------------
{'email': 'Production is DOWN, customers cannot check out!', 'urgency': 'high'}
+ a two-node picture: __start__ -> classify -> __end__
"""

### Pattern 2 · Prompt chain
**Use case: process a supplier invoice. *extract → validate → post.***

The model fills in each step, and never "chooses" the next step.

Predictable, and auditable. Suitable for high-volume, well-specified pipelines.

Note *which* step gets the model: only `extract`, the fuzzy one. `validate` is an arithmetic
comparison and `post` is a write — paying a model to do either would buy you nothing but variance.
That split (model for judgement, code for rules) is the single most reusable habit in this notebook.


In [ ]:
# Pattern 2 - PROMPT CHAIN.  Use case: process a supplier invoice: extract -> validate -> post.
# YOU wrote the order of steps; the model only fills each one in.
from pydantic import BaseModel, Field

class InvoiceState(TypedDict):
    raw: str
    fields: dict
    valid: bool
    posted: bool

RAW_INVOICE = """ACME Northgate Supplies Ltd
Invoice INV-88213   PO Reference: PO-4471
Line total ................ 1,290.00 USD"""

class InvoiceFields(BaseModel):
    """The schema we force the model's answer into. with_structured_output() sends this to the
    provider as a function/tool schema, so we get a validated object back instead of prose that
    we then have to regex. Use it every time you need machine-readable output from a model."""
    po: str      = Field(description="purchase order reference, e.g. PO-1234")
    amount: float = Field(description="invoice total as a number, no currency symbol or commas")
    vendor: str  = Field(description="vendor name, one or two words")

def extract(state: InvoiceState) -> dict:          # step 1: pull the fixed field set from the invoice
    if LLM_ENABLED:
        try:
            parsed = chat_model.with_structured_output(InvoiceFields).invoke(
                [SystemMessage(content="Extract the invoice fields exactly as they appear."),
                 HumanMessage(content=state["raw"])])
            return {"fields": parsed.model_dump()}
        except Exception as e:
            print(f"  [structured extraction failed: {type(e).__name__}] using fixture")
    return {"fields": {"po": "PO-4471", "amount": 1290.0, "vendor": "Northgate"}}

def validate(state: InvoiceState) -> dict:         # step 2: a rule. No model - it is arithmetic.
    return {"valid": state["fields"]["amount"] < 5000}

def post(state: InvoiceState) -> dict:             # step 3: the write to the ERP
    return {"posted": state["valid"]}

g = StateGraph(InvoiceState)
g.add_node("extract", extract)
g.add_node("validate", validate)
g.add_node("post", post)
g.add_edge(START, "extract")
g.add_edge("extract", "validate")
g.add_edge("validate", "post")
g.add_edge("post", END)
prompt_chain = g.compile()

result = prompt_chain.invoke({"raw": RAW_INVOICE, "fields": {}, "valid": False, "posted": False})
print("extracted:", result["fields"])
print("valid:", result["valid"], "| posted:", result["posted"])
show_graph(prompt_chain, "Pattern 2 · prompt chain")

"""
EXPECTED OUTPUT
---------------
extracted: {'po': 'PO-4471', 'amount': 1290.0, 'vendor': ...}      <- vendor wording may vary
valid: True | posted: True
+ a straight-line picture: __start__ -> extract -> validate -> post -> __end__
"""

### Pattern 3 · Router
**Use case: IT helpdesk ticket triage.**

The model makes exactly **one run-time choice**: which of N known branches it should take. Each branch is a path you define. The routing function is a pure function of state, so you can even unit-test the routing without a graph or a model.


In [ ]:
# Pattern 3 - ROUTER.  Use case: IT helpdesk triage. The model picks ONE of N enumerated paths.
CATEGORIES = ["vpn", "password", "other"]

class TicketState(TypedDict):
    text: str
    category: str
    action: str

def classify(state: TicketState) -> dict:
    """The model DECIDES here, inside a node, and writes its decision into state."""
    label = ask(system=f"Classify the IT ticket into exactly one of: {', '.join(CATEGORIES)}. "
                       f"Reply with the single word only.",
                user=state["text"])
    label = (label or "").lower().strip(" .!\n")
    if label not in CATEGORIES:                                   # keyword fallback
        t = state["text"].lower()
        label = "vpn" if "vpn" in t else "password" if "password" in t else "other"
    return {"category": label}

def route(state: TicketState) -> str:
    """The edge only READS the decision - no model call in here, ever.
    That is why the one run-time decision in this graph is unit-testable in one line."""
    return state["category"]

def do_vpn(state):      return {"action": "ran vpn_reconnect script"}
def do_password(state): return {"action": "ran password_reset script"}
def do_other(state):    return {"action": "escalated to human queue"}

g = StateGraph(TicketState)
g.add_node("classify", classify)
g.add_node("vpn", do_vpn)
g.add_node("password", do_password)
g.add_node("other", do_other)
g.add_edge(START, "classify")
# The 3rd argument maps the router's RETURN VALUES to node names. Returning a value that is not a
# key here is the most common wiring bug in LangGraph - and it raises at run time, not compile time.
g.add_conditional_edges("classify", route,
                        {"vpn": "vpn", "password": "password", "other": "other"})
for n in CATEGORIES:
    g.add_edge(n, END)
router = g.compile()

print(router.invoke({"text": "My VPN will not connect from home", "category": "", "action": ""}))
assert route({"text": "", "category": "vpn", "action": ""}) == "vpn"   # no graph, no model, no tokens
show_graph(router, "Pattern 3 · router")

"""
EXPECTED OUTPUT
---------------
{'text': 'My VPN will not connect from home', 'category': 'vpn', 'action': 'ran vpn_reconnect script'}
+ a fan-out picture: classify branches (dotted lines) to vpn / password / other
"""

### Pattern 4 · Single tool-calling agent
**Use case: supplier risk research.**

The **model** chooses which tool to call and when to stop. The graph is a **loop**, and the exact trajectory only exists at run time. You have crossed from workflow into *agent*. This means that we now have flexibility, but potentially at the price of determinism and testability.

Three new pieces appear here, and they carry through the rest of the notebook:

* **`@tool`** turns a Python function into something the model can see. Its *docstring is the prompt*
  the model reads when deciding whether to call it — a vague docstring is a vague agent.
* **`.bind_tools([...])`** attaches those schemas to the model, so its reply may now contain
  `tool_calls` instead of text.
* **`Annotated[list, add_messages]`** is the conversation reducer: nodes *append* messages rather than
  overwrite them, which is how the agent remembers what it already tried.


In [ ]:
# Pattern 4 - SINGLE (TOOL-CALLING) AGENT.  Use case: supplier risk research.
# The model decides the next tool AND when to stop; the graph loops until it stops asking.
from langchain_core.tools import tool
from langgraph.graph import add_messages

@tool
def news_search(supplier: str) -> str:
    """Search recent news for adverse coverage about a supplier."""
    return f"{supplier}: no adverse news in the last 12 months."

@tool
def sanctions_list(supplier: str) -> str:
    """Check a supplier against sanctions and watchlists."""
    return f"{supplier}: not present on any sanctions list."

@tool
def past_contracts(supplier: str) -> str:
    """Look up our own contract history with a supplier."""
    return f"{supplier}: 2 prior contracts, no disputes."

RESEARCH_TOOLS = [news_search, sanctions_list, past_contracts]
TOOLS_BY_NAME  = {t.name: t for t in RESEARCH_TOOLS}
MAX_TOOL_STEPS = 6          # a cost/latency ceiling: an agent loop with no cap is an open invoice

class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]

def agent(state: ResearchState) -> dict:
    """The policy node: look at everything so far, decide to call a tool or to answer."""
    if LLM_ENABLED:
        model = chat_model.bind_tools(RESEARCH_TOOLS)
        return {"messages": [model.invoke(state["messages"])]}
    # Fallback policy: same message shapes, scripted decisions - the GRAPH is identical either way.
    used = {m.name for m in state["messages"] if isinstance(m, ToolMessage)}
    todo = [t.name for t in RESEARCH_TOOLS if t.name not in used]
    if todo:
        return {"messages": [AIMessage(content="", tool_calls=[
            {"name": todo[0], "args": {"supplier": "Northgate Pay"}, "id": f"call_{todo[0]}"}])]}
    return {"messages": [AIMessage(content="Northgate Pay: low risk (3 sources checked).")]}

def tools_node(state: ResearchState) -> dict:
    """Execute every tool the model asked for. langgraph.prebuilt.ToolNode does exactly this -
    we hand-roll it once so the mechanism is not magic (see the ReAct cell for the prebuilt)."""
    calls = state["messages"][-1].tool_calls
    return {"messages": [
        # name= is optional for the provider but makes the observation self-describing when you
        # (or a fallback policy) read the message list back.
        ToolMessage(content=TOOLS_BY_NAME[c["name"]].invoke(c["args"]),
                    tool_call_id=c["id"], name=c["name"])
        for c in calls]}

def keep_going(state: ResearchState) -> str:
    """Continuation is decided by the MODEL (did it ask for a tool?), bounded by YOU (the cap)."""
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None) and len(state["messages"]) < MAX_TOOL_STEPS * 2:
        return "tools"
    return END

g = StateGraph(ResearchState)
g.add_node("agent", agent)
g.add_node("tools", tools_node)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", keep_going, {"tools": "tools", END: END})
g.add_edge("tools", "agent")            # <- the loop: observe, then think again
single_agent = g.compile()

out = single_agent.invoke({"messages": [
    SystemMessage(content="You assess supplier risk. Use every tool available to you exactly once, "
                          "then write a one-sentence risk memo."),
    HumanMessage(content="Assess the supplier 'Northgate Pay'.")]})

for m in out["messages"]:
    kind = type(m).__name__
    detail = getattr(m, "tool_calls", None) or (m.content[:70] if m.content else "")
    print(f"  {kind:<12} {detail}")
show_graph(single_agent, "Pattern 4 · single agent (tool loop)")

"""
EXPECTED OUTPUT
---------------
A message trace, then a cyclic picture (agent <-> tools). The exact number of turns and the memo
wording vary by model, but the SHAPE must be:
  SystemMessage / HumanMessage
  AIMessage    [{'name': 'news_search', ...}]      <- model asks for a tool
  ToolMessage  Northgate Pay: no adverse news...   <- observation goes back into the messages
  ... (repeats) ...
  AIMessage    Northgate Pay: low risk ...         <- no tool_calls => the loop ends
"""

### Pattern 5 · Multi-agent
**Use case: enterprise contract review.** Three specialists with **different expertise** run in parallel; a merge node joins their reads.

More agents are great for parallelism and separated expertise. However, multi-agent coordination can be costly, where information must be re-serialised. In **Session 2, we will build a full supervisor team in this shape with real LLM calls.**


In [ ]:
# Pattern 5 - MULTI-AGENT.  Use case: enterprise contract review.
# legal / finance / compliance fan OUT (run in parallel), then a merge node fans IN.
CONTRACT = """MASTER SERVICES AGREEMENT (extract)
7. Liability. Supplier's aggregate liability is unlimited for any breach of this agreement.
9. Payment. Fees are payable net-90 from invoice date, in advance of delivery.
14. Data. Customer data may be processed in any jurisdiction at Supplier's discretion."""

class ContractState(TypedDict):
    contract: str
    reads: Annotated[list, add]     # each specialist APPENDS; the reducer merges concurrent writes
    memo: str

def specialist(role: str, brief: str, fallback: str):
    """One factory, three agents. Each gets a NARROW system prompt - that scoping IS the
    'specialisation'; there is no other magic in a multi-agent system."""
    def node(state: ContractState) -> dict:
        finding = ask(system=f"You are a {role} reviewer. {brief} "
                             f"Reply with ONE sentence naming the clause number and the risk.",
                      user=state["contract"])
        return {"reads": [f"{role}: {finding or fallback}"]}
    return node

legal      = specialist("legal", "You care about liability, indemnity and termination.",
                        "clause 7 unlimited liability exceeds playbook risk")
finance    = specialist("finance", "You care about payment terms and cash exposure.",
                        "clause 9 net-90 in advance, exposure high")
compliance = specialist("compliance", "You care about data residency and sanctions.",
                        "clause 14 unrestricted jurisdiction breaks data residency")

def merge(state: ContractState) -> dict:
    return {"memo": "\n  ".join(sorted(state["reads"]))}

g = StateGraph(ContractState)
for name, fn in [("legal", legal), ("finance", finance), ("compliance", compliance), ("merge", merge)]:
    g.add_node(name, fn)
for spec in ("legal", "finance", "compliance"):
    g.add_edge(START, spec)         # fan OUT - all three specialists start in the SAME superstep
    g.add_edge(spec, "merge")       # fan IN  - merge waits for all three to finish
g.add_edge("merge", END)
multi_agent = g.compile()

result = multi_agent.invoke({"contract": CONTRACT, "reads": [], "memo": ""})
print("merged memo:\n  " + result["memo"])
# Three nodes wrote 'reads' in one superstep. Without Annotated[list, add] that is an
# InvalidUpdateError, not a silent overwrite - LangGraph refuses to guess. (Proof in Part B1.)
show_graph(multi_agent, "Pattern 5 · multi-agent (parallel specialists + merge)")

"""
EXPECTED OUTPUT
---------------
merged memo:
  compliance: ...clause 14...
  finance: ...clause 9...
  legal: ...clause 7...
+ a diamond picture: __start__ fans out to three specialists, all three fan in to merge
(Wording varies with the model; what must hold is THREE entries, one per specialist.)
"""

---
## Part A3 · The named agentic patterns

Patterns 1–4 are one graph each. Pattern 5 is a family, and the family members have names you will
meet in every agent codebase and every paper. Two vocabularies describe the same space — the
**topology** (what the wiring looks like) and the **named pattern** (what the wiring is *for*):

| Topology | Shape | Use when |
|---|---|---|
| **Map-reduce** | fan out → merge | Subtasks are independent, then joined. *(Pattern 5 above.)* |
| **Verification** | generator → verifier → repair | Correctness is checkable by code. **Prefer this whenever it applies.** *(Lab B.)* |
| **Critic-revision** | generator ↔ critic loop | Output improves with rounds, and "good enough" is judged, not computed. *(Reflection, below.)* |
| **Supervisor** | a router agent delegates | Which specialist is needed depends on the input. *(Supervisor-Worker, below; full team in Session 2.)* |
| **Debate** | N agents argue → judge | Correctness is contested and no checker exists. Expensive; rarely the first answer. |
| **Hierarchical / swarm** | managers of managers / no manager | Large teams. Almost never the right starting point. |

The four cells that follow are the named patterns, each as a runnable graph:

| Pattern | One-line definition | Reach for it when |
|---|---|---|
| **ReAct** | Interleave **Rea**soning and **Act**ing: think → call a tool → observe → think again | The next action depends on what the last one returned |
| **Planner-Executor** | Write the whole plan first, then execute it step by step | The work is decomposable up-front and you want the plan reviewable (or human-approvable) before spending |
| **Reflection** | Generate, critique your own output, revise, repeat under a cap | Quality improves with rounds and no deterministic checker exists |
| **Supervisor-Worker** | A supervisor agent routes work to specialists and decides when to stop | Distinct expertise is genuinely needed and the routing is data-dependent |

They compose: a supervisor whose workers are ReAct agents, one of which reflects, is an ordinary
production design. Build them one at a time and only combine when a single pattern demonstrably fails.


### Pattern · ReAct (reason → act → observe)

ReAct is the default shape of a modern tool-using agent, and Pattern 4 above was already one: the
model *reasons* about what it needs, *acts* by calling a tool, *observes* the result in the message
list, and reasons again with that new information. The loop exists precisely because the second
action cannot be chosen until the first observation exists — that is the irreducible ambiguity that
justifies an agent over a workflow.

Here we build it the way you would in production: with LangGraph's **prebuilt `ToolNode`** (which
executes tool calls, including several in parallel, and formats errors back to the model) and
**`tools_condition`** (the standard "did the model ask for a tool?" edge). The scenario is an on-call
assistant diagnosing an alert — the canonical case where the next lookup depends on the last finding.


In [ ]:
# ReAct = the agent<->tools cycle, built from LangGraph's prebuilts instead of by hand.
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def get_recent_deploys(service: str) -> str:
    """List deployments to a service in the last 24 hours, newest first."""
    return "14:02 checkout-api v2.31 (config change: DB_POOL_SIZE 50->5); 09:10 checkout-api v2.30"

@tool
def search_logs(service: str, query: str) -> str:
    """Search a service's error logs for a substring. Returns matching lines with counts."""
    return "1,204 x 'TimeoutError: connection pool exhausted' since 14:05"

@tool
def get_metric(service: str, metric: str) -> str:
    """Read a named metric for a service over the last hour (p99_latency, error_rate, cpu)."""
    return f"{metric}: flat until 14:05, then 30x above baseline"

ONCALL_TOOLS = [get_recent_deploys, search_logs, get_metric]
ONCALL_BY_NAME = {t.name: t for t in ONCALL_TOOLS}
REACT_STEP_CAP = 8

class OnCallState(TypedDict):
    messages: Annotated[list, add_messages]

# Scripted trajectory for the no-key path: a plausible reason->act sequence, so the graph, the
# ToolNode and the printed trace all behave identically whether or not a model is available.
FALLBACK_TRAJECTORY = [
    ("I should first check whether anything was deployed around the time the alert fired.",
     "get_recent_deploys", {"service": "checkout-api"}),
    ("A deploy at 14:02 changed the DB pool size; let me confirm the error signature in the logs.",
     "search_logs", {"service": "checkout-api", "query": "pool"}),
    ("Now I want to see whether latency moved at the same moment.",
     "get_metric", {"service": "checkout-api", "metric": "p99_latency"}),
]

def react_agent(state: OnCallState) -> dict:
    if LLM_ENABLED:
        return {"messages": [chat_model.bind_tools(ONCALL_TOOLS).invoke(state["messages"])]}
    step = sum(1 for m in state["messages"] if isinstance(m, ToolMessage))
    if step < len(FALLBACK_TRAJECTORY):
        thought, name, args = FALLBACK_TRAJECTORY[step]
        return {"messages": [AIMessage(content=thought,
                                       tool_calls=[{"name": name, "args": args, "id": f"c{step}"}])]}
    return {"messages": [AIMessage(content=(
        "Root cause: the 14:02 deploy of checkout-api v2.31 cut DB_POOL_SIZE from 50 to 5, "
        "exhausting the connection pool and driving p99 latency 30x above baseline. "
        "Recommended action: roll back v2.31."))]}

def react_continue(state: OnCallState) -> str:
    """tools_condition alone would loop forever against a stubborn model, so we AND it with a cap.
    Every agent loop you ship needs a bound that does not depend on the model's cooperation."""
    if sum(1 for m in state["messages"] if isinstance(m, ToolMessage)) >= REACT_STEP_CAP:
        return END
    return tools_condition(state)      # returns "tools" if the last AIMessage has tool_calls, else END

g = StateGraph(OnCallState)
g.add_node("agent", react_agent)
g.add_node("tools", ToolNode(ONCALL_TOOLS))     # the prebuilt version of Pattern 4's tools_node
g.add_edge(START, "agent")
g.add_conditional_edges("agent", react_continue, {"tools": "tools", END: END})
g.add_edge("tools", "agent")
react_agent_graph = g.compile()

out = react_agent_graph.invoke({"messages": [
    SystemMessage(content="You are an on-call SRE. Diagnose the alert by calling tools one at a "
                          "time. Before each call, state your reasoning in one short sentence. "
                          "When you know the root cause, answer without calling any tool."),
    HumanMessage(content="ALERT: checkout-api p99 latency breached SLO at 14:05.")]})

print("=== the ReAct trace ===")
for m in out["messages"][2:]:
    if isinstance(m, AIMessage):
        if m.content:                       print(f"  THOUGHT   {m.content}")
        for c in (m.tool_calls or []):      print(f"  ACT       {c['name']}({c['args']})")
    elif isinstance(m, ToolMessage):        print(f"  OBSERVE   {m.content}")
show_graph(react_agent_graph, "ReAct · agent <-> tools")

"""
EXPECTED OUTPUT
---------------
An alternating trace - THOUGHT / ACT / OBSERVE / THOUGHT / ... ending in a THOUGHT with no ACT -
and a two-node cyclic picture (agent <-> tools, with a dotted edge from agent to __end__).
The number of tool calls and the wording vary with the model; the alternation does not.
"""

### Pattern · Planner-Executor

ReAct decides one step at a time. Planner-Executor decides **all** the steps first, then executes
them. You trade adaptivity for two things you often want more: the plan is a concrete artifact you
can log, cost-estimate, or put in front of a human before any money is spent, and each executor step
runs against a short, focused context instead of an ever-growing transcript.

The trap is a stale plan: if step 2's result invalidates step 3, a rigid executor marches on
regardless. The production compromise is **replanning** — after each step, ask whether the remaining
plan still makes sense. The graph below leaves the replan edge in place as a commented-out
one-liner, so you can see exactly where the choice lives.


In [ ]:
# Planner-Executor: plan once (a real artifact), then execute the plan step by step.
class PlanState(TypedDict):
    goal: str
    plan: list[str]        # remaining steps - the executor pops from the front
    done: list[str]        # (step, result) pairs already executed
    report: str

FALLBACK_PLAN = [
    "Pull the last 4 quarters of churn numbers by customer segment",
    "Identify the two segments with the steepest churn increase",
    "Summarise the likely drivers and recommend one retention action",
]

def planner(state: PlanState) -> dict:
    """One model call produces the WHOLE plan. Parsing it into a typed list at this boundary is
    what makes the rest of the graph ordinary Python."""
    raw = ask(system="You are a planner. Break the goal into 3 short, concrete, ordered steps. "
                     "Reply with one step per line, no numbering, no commentary.",
              user=state["goal"])
    steps = [l.strip(" -*0123456789.") for l in (raw or "").splitlines() if l.strip()]
    return {"plan": steps[:3] if len(steps) >= 2 else FALLBACK_PLAN}

def executor(state: PlanState) -> dict:
    """Executes exactly ONE step per visit, then returns to the router. Doing the whole plan inside
    one node would work, but you would lose a checkpoint (and a resume point) per step."""
    step, rest = state["plan"][0], state["plan"][1:]
    result = ask(system="You execute one analysis step and report the outcome in one sentence. "
                        "Invent plausible figures - this is a lab.",
                 user=f"Goal: {state['goal']}\nStep: {step}") or f"(simulated result for: {step})"
    return {"plan": rest, "done": state["done"] + [f"{step} -> {result}"]}

def plan_router(state: PlanState) -> str:
    return "executor" if state["plan"] else "report"
    # Replanning variant: return "planner" here when the last result contradicts the remaining plan.
    # That single edge turns Planner-Executor into a plan-and-adapt agent - and costs one call per step.

def reporter(state: PlanState) -> dict:
    return {"report": " | ".join(state["done"])}

g = StateGraph(PlanState)
g.add_node("planner", planner)
g.add_node("executor", executor)
g.add_node("report", reporter)
g.add_edge(START, "planner")
g.add_conditional_edges("planner", plan_router, {"executor": "executor", "report": "report"})
g.add_conditional_edges("executor", plan_router, {"executor": "executor", "report": "report"})
g.add_edge("report", END)
planner_executor = g.compile()

out = planner_executor.invoke({"goal": "Explain why enterprise churn rose last quarter",
                               "plan": [], "done": [], "report": ""})
print("THE PLAN, AND WHAT EACH STEP RETURNED:")
for i, d in enumerate(out["done"], 1):
    print(f"  {i}. {d}")
print("\nsteps executed:", len(out["done"]), "| plan remaining:", out["plan"])
show_graph(planner_executor, "Planner-Executor · plan once, execute step by step")

"""
EXPECTED OUTPUT
---------------
Three numbered lines of the form "<step> -> <result>" (the plan was written in ONE call,
before any of them ran), then:
  steps executed: 3 | plan remaining: []
+ a picture where BOTH planner and executor route (dotted) to executor or report.
(The steps themselves are written by the model, so their wording changes every run.)
"""

### Pattern · Reflection (generate → critique → revise)

Reflection adds a second model pass whose only job is to attack the first one's output, and then a
third that repairs it. It measurably helps on writing, code and analysis — but note what it is and is
not. The critic shares the generator's blind spots, because it is the same model with the same
training; **self-critique catches sloppiness, not ignorance**.

So the rule is: if a deterministic checker exists, use *that* as the critic (compiler, schema
validator, unit test, policy rule — exactly what `check_document()` does in Lab B) and keep the model
for the repair. Reflection with a model critic is the fallback for when no such oracle exists, and it
always needs a round cap, because "the critic is satisfied" is not a condition you control.


In [ ]:
# Reflection: draft -> critique -> revise, bounded by MAX_ROUNDS.
MAX_REFLECT_ROUNDS = 2
RULES = "Must be under 60 words, must name a concrete number, must end with a clear ask."

class ReflectState(TypedDict):
    brief: str
    draft: str
    critique: str
    rounds: int

def generate(state: ReflectState) -> dict:
    draft = ask(system=f"Write a short internal announcement. Rules: {RULES}",
                user=state["brief"])
    return {"draft": draft or "We are moving to the new expense tool. Please switch over soon."}

def critique(state: ReflectState) -> dict:
    """The critic gets the RULES and the draft - and nothing else. A critic that also sees the
    generator's reasoning tends to agree with it; independence is the entire value of this node."""
    verdict = ask(system=f"You review internal announcements against these rules: {RULES} "
                         f"If the draft satisfies every rule reply exactly PASS. "
                         f"Otherwise reply with the single most important fix, in one sentence.",
                  user=state["draft"])
    if verdict is None:                      # deterministic stand-in for the critic
        verdict = "PASS" if len(state["draft"].split()) < 60 and any(
            ch.isdigit() for ch in state["draft"]) else "Add the deadline date and a concrete number."
    return {"critique": verdict.strip()}

def revise(state: ReflectState) -> dict:
    new = ask(system=f"Rewrite the announcement applying the reviewer's fix. Rules: {RULES} "
                     f"Return only the rewritten text.",
              user=f"DRAFT:\n{state['draft']}\n\nFIX:\n{state['critique']}")
    fallback = ("We move to the new expense tool on 30 June. Submit anything older in the current "
                "tool before then - it takes about 5 minutes. Set up your new account this week.")
    return {"draft": new or fallback, "rounds": state["rounds"] + 1}

def reflect_router(state: ReflectState) -> str:
    """Two exit conditions, and you need both: the critic is happy, OR you have spent enough.
    Without the second, a fastidious critic bills you forever."""
    if state["critique"].upper().startswith("PASS"):   return "done"
    if state["rounds"] >= MAX_REFLECT_ROUNDS:          return "done"
    return "revise"

g = StateGraph(ReflectState)
g.add_node("generate", generate)
g.add_node("critique", critique)
g.add_node("revise", revise)
g.add_edge(START, "generate")
g.add_edge("generate", "critique")
g.add_conditional_edges("critique", reflect_router, {"revise": "revise", "done": END})
g.add_edge("revise", "critique")          # every revision is re-critiqued - that is the loop
reflection = g.compile()

out = reflection.invoke({"brief": "Announce that the company is switching expense tools next month.",
                         "draft": "", "critique": "", "rounds": 0})
print("rounds of revision :", out["rounds"], f"(cap {MAX_REFLECT_ROUNDS})")
print("last critique      :", out["critique"])
print("final draft        :", out["draft"])
show_graph(reflection, "Reflection · generate -> critique -> revise")

"""
EXPECTED OUTPUT
---------------
rounds of revision : 0, 1 or 2 - never more than the cap
last critique      : either PASS, or the fix that was still outstanding when the cap hit
final draft        : the revised announcement text
+ a picture with a critique <-> revise cycle and a dotted edge from critique to __end__
"""

### Pattern · Supervisor-Worker

A supervisor is a router that runs in a loop: it looks at the shared state, picks the specialist who
should act next, and decides when the work is finished. That is the difference from Pattern 3's
router — a router chooses once, a supervisor chooses repeatedly with the accumulated results in view.

The costs are real and worth stating before you reach for it: one extra model call per hop (the
supervisor's own turn), and a lossy boundary at every handoff, because each worker sees a summary of
what came before rather than the original context. Reach for it only when the specialists genuinely
differ in expertise or tools **and** you cannot predict the order in advance.

Two failure modes to design against, both visible in the code below: the supervisor picking a worker
forever (fix: a hop cap), and the supervisor naming a worker that does not exist (fix: validate its
answer against the roster before it reaches the edge). **Session 2 builds the full five-agent version
of this, with reviewer loop-backs.**


In [ ]:
# Supervisor-Worker: one routing agent, three specialists, a loop, and a hop cap.
WORKERS  = ["researcher", "writer", "editor"]
MAX_HOPS = 6

class TeamState(TypedDict):
    task: str
    notes: Annotated[list, add]     # every worker APPENDS its contribution to the shared record
    next: str                       # the supervisor's decision, written into state (never into the edge)
    hops: int

def supervisor(state: TeamState) -> dict:
    done = [n.split(":")[0] for n in state["notes"]]
    choice = ask(
        system=f"You supervise a team: {', '.join(WORKERS)}. Given the task and what has already "
               f"been done, reply with the name of the ONE worker who should act next, or FINISH "
               f"if the task is complete. Reply with a single word.",
        user=f"TASK: {state['task']}\nALREADY DONE: {done or 'nothing yet'}")
    choice = (choice or "").lower().strip(" .!\n")
    if choice not in WORKERS + ["finish"]:
        # Never trust a free-text routing decision. Fall back to a deterministic order so a
        # hallucinated worker name degrades into a sensible default instead of a KeyError.
        remaining = [w for w in WORKERS if w not in done]
        choice = remaining[0] if remaining else "finish"
    return {"next": choice, "hops": state["hops"] + 1}

def worker(role: str, brief: str, fallback: str):
    def node(state: TeamState) -> dict:
        out = ask(system=f"You are the {role} on a content team. {brief} Reply in one sentence.",
                  user=f"TASK: {state['task']}\nTEAM NOTES SO FAR:\n" + "\n".join(state["notes"]))
        return {"notes": [f"{role}: {out or fallback}"]}
    return node

def supervisor_router(state: TeamState) -> str:
    if state["next"] == "finish" or state["hops"] >= MAX_HOPS:
        return END
    return state["next"]

g = StateGraph(TeamState)
g.add_node("supervisor", supervisor)
g.add_node("researcher", worker("researcher", "You gather the two facts that matter most.",
                                "two comparable rollouts took 6 and 9 weeks"))
g.add_node("writer",     worker("writer", "You draft the content from the researcher's facts.",
                                "drafted a 3-paragraph rollout plan"))
g.add_node("editor",     worker("editor", "You tighten the draft and flag anything unsupported.",
                                "cut 40% of the draft, flagged one unsourced claim"))
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", supervisor_router,
                        {w: w for w in WORKERS} | {END: END})
for w in WORKERS:
    g.add_edge(w, "supervisor")        # every worker reports BACK to the supervisor - the star shape
supervisor_team = g.compile()

out = supervisor_team.invoke({"task": "Produce a one-page brief on migrating our CRM.",
                              "notes": [], "next": "", "hops": 0})
print("supervisor hops :", out["hops"], f"(cap {MAX_HOPS})")
for n in out["notes"]:
    print("  " + n)
show_graph(supervisor_team, "Supervisor-Worker · star topology with loop-back")

"""
EXPECTED OUTPUT
---------------
supervisor hops : 4 or fewer (3 workers + the FINISH turn), never more than the cap
one line per worker that acted, in the order the supervisor chose them
+ a STAR picture: supervisor at the centre, dotted edges out to each worker, solid edges back
"""

---
## Part A4 · The decision tree

![image.png](https://drive.google.com/uc?id=1YkN2-1Ypc4tR70-6Ynh32HE1LXHZCAnM)



**Q1 is the important one.** "Enumerable" means *you* can write the paths down now — not that the task is easy. If you can write them down, do; a workflow is cheaper, testable and reproducible.

**Q2 defaults to YES.** A second agent adds a boundary where context is re-serialised and information is lost. Split only for a reason you can name.

### Worksheet — six problems

Walk each one down the tree, then justify it. **The justification column is the point.**

| # | Problem |
|---|---|
| W1 | Redact personal data from 200,000 scanned patient intake forms before analytics. |
| W2 | An inbound insurance claim email must be sorted into one of six claim types and sent down that type's processing path. |
| W3 | An on-call assistant diagnoses a production alert: it reads logs, metrics and recent deploys, and what it looks at next depends on what it just found. |
| W4 | Turn a plain-English network change request into a vendor-specific config, then check it for syntax, schema and policy violations and repair it until it passes. |
| W5 | A quarterly vendor due-diligence pack: for each vendor, a financial, a legal and a security review, each written by someone who knows that domain, then merged into one recommendation. |
| W6 | An internal "ask anything" assistant for a bank: some questions are HR policy lookups, some need live account data, some need a compliance-approved answer — and a single question may need two of those before it can be answered. |

Fill this in (double-click to edit):

| # | Pattern | If multi-agent, which topology | Justification: name the cheaper pattern, and the one fact that rules it out |
|---|---|---|---|
| W1 | | | |
| W2 | | | |
| W3 | | | |
| W4 | | | |
| W5 | | | |
| W6 | | | |

**What a justification looks like.** Not "it needs to be smart." Instead: *"Pattern 4, not pattern 3 — the next lookup depends on what the last one returned, so the paths cannot be enumerated at design time."*


<details>
<summary><b>Solutions — open only after you have written your own answers</b></summary>

| # | Pattern | Topology / named pattern | Justification |
|---|---|---|---|
| W1 | **1 · Single LLM call** | — | One input → one output, no branching, 200,000 times. The only real questions are batching and cost per page. Anything above pattern 1 here is paying for orchestration that has nothing to orchestrate. |
| W2 | **3 · Router** | — | The six claim types are *enumerated in the spec*, so every path is known at design time. Not pattern 4: the model makes exactly one decision and never chooses what to do next. Not pattern 2 either — a chain has no branch. |
| W3 | **4 · Single agent** | ReAct | The next lookup depends on what the last one returned, so the paths cannot be enumerated. Not pattern 3, for exactly that reason. Not multi-agent: one context, one skill set, nothing to parallelise — a second agent would only add a lossy handoff. |
| W4 | **5 · Multi-agent** | Verification (generate → check → repair) | A deterministic oracle exists — the syntax parser, the schema and the policy engine — so the critic must be *code*, not a model. Prefer this over Reflection whenever a checker exists; the loop is bounded by "it compiles", not by a model's opinion. This is Lab B's shape. |
| W5 | **5 · Multi-agent** | Map-reduce (fan-out → merge) | Three genuinely distinct expertises with no dependency between them, so they run in parallel and the merge is a real join. Not a supervisor: nothing is data-dependent — you always need all three, so a routing agent would be an extra model call that decides nothing. |
| W6 | **5 · Multi-agent** | Supervisor-Worker | Which specialist is needed depends on the question, a question can need two of them, and the order is not knowable in advance — that combination is precisely what a supervisor loop buys. A plain router (pattern 3) fails on the "needs two" case: it chooses once. Cap the hops. |

**The pattern to notice:** four of the six are solved *below* the top of the ladder, and the two that
are not each name a specific capability the cheaper pattern cannot deliver. That is the entire
discipline — a pattern choice you cannot argue against is a pattern choice you have not made.

</details>


---
### Lab A wrap

You should now be able to place a problem on the tree, and argue **against** your own choice by naming the cheaper pattern and the single fact that rules it out.

**Discussion checkpoint — with your TA:**

1. Someone proposes a five-agent team for W2. Give your answer and your reasoning.
2. In the multi-agent cell, the specialists' findings are re-serialised at `merge`. What is lost there that no amount of shared state can preserve?
3. W4 could be built with Reflection (a model critic) instead of Verification (a code checker). Name what you would lose.

**Optional.** Take a system you need at work, place it on the tree, and sketch it with `show_graph`.


---
---
# LAB B · Document Approval Workflow in LangGraph

*Deliverable: a checkpointed state graph that pauses for a human and resumes after a kernel
restart. This is **Milestone 5** of your capstone.*

Lab A ended with a conclusion: **most business processes do not need agency: they need durability
and a gate.** This lab builds exactly that, and it is deliberately a *workflow*, not a full agent. Every
routing decision here is made by you, in a pure Python function you can unit-test.

Target process: `draft → automated checks → conditional routing → human approval interrupt → finalize`,
checkpointed so it can pause for a human and resume **after the kernel has been restarted**.

One node does call a model — the `revise` step, because rewriting prose is the only genuinely fuzzy
part of this process. Everything that *decides* anything stays deterministic, which is what keeps the
whole workflow testable. That division is the lesson of Lab B as much as checkpointing is.


---
## Part B1 · State: the most important decision in the graph

Every LangGraph graph operates on one **state** object that flows through every node. Two rules, both non-obvious:

**Rule 1: a node returns a *partial* update, not the whole state.** Returning `{"issues": [...]}` updates only that key.

**Rule 2: the default merge is *overwrite*; a reducer changes that.** A field annotated `Annotated[list, add]` accumulates across writes; an unannotated field is replaced by the latest writer.

> **Analogy.** A reducer is the merge strategy in version control. No reducer = force-push: last writer wins and the other branch's work disappears without a conflict marker. A reducer is the merge driver you register so two writes combine instead of one silently vanishing.

### The design trap in this specific workflow

It is tempting to accumulate `issues` with a reducer so you keep the full history. **Do not.** Your router will read `issues` to decide whether to send the document back for revision — and if `issues` accumulates, it is *never* empty after the first failure, so the document loops forever.

The fix is the general principle: **separate facts from control.**

| Field | Reducer? | Why |
|---|---|---|
| `issues` | **no** — overwrite | It is a *control* field. It must describe the document as it is **now**. |
| `issue_log` | **yes** — append | It is an *audit* field. It must describe everything that was ever wrong. |

Two fields instead of one is not redundancy. It is the difference between a workflow that terminates and one that does not.

In [ ]:
# The state schema - the most consequential ten lines in any LangGraph project.
# Keep it minimal: every field here is serialised into every checkpoint. If data does not
# cross a node boundary, it is a local variable, not state.
from typing import TypedDict, Annotated
from operator import add

MAX_REVISIONS = 3

class ReviewState(TypedDict):
    doc_id: str
    draft: str                 # overwritten - the latest version is the only one that matters
    issues: list[str]          # CONTROL field: current issues only. NO reducer. (see the table above)
    revision_count: int        # overwritten - a plain counter used as the loop guard
    status: str                # CONTROL field read by the routers
    approver_note: str

    # AUDIT field. Annotated[..., add] registers a REDUCER: when a node returns a value for this
    # key, LangGraph calls add(existing, new) instead of replacing. For a list that is concatenation,
    # so the trail accumulates across revisions. Other common reducers: operator.or_ for dicts,
    # langgraph.graph.add_messages for chat history, or any 2-arg function you write yourself.
    issue_log: Annotated[list[str], add]

def has_reducer(field_name: str) -> bool:
    """Annotated[...] stashes its extras in __metadata__ - that is how the framework finds them."""
    return hasattr(ReviewState.__annotations__[field_name], "__metadata__")

print("state fields:", list(ReviewState.__annotations__))
print("issues    has reducer:", has_reducer("issues"))
print("issue_log has reducer:", has_reducer("issue_log"))

"""
EXPECTED OUTPUT
---------------
state fields: ['doc_id', 'draft', 'issues', 'revision_count', 'status', 'approver_note', 'issue_log']
issues    has reducer: False
issue_log has reducer: True
"""

In [ ]:
# Self-check B1 — the reducer must be on the audit field and NOT on the control field.
ann = ReviewState.__annotations__
assert hasattr(ann["issue_log"], "__metadata__"), \
    "issue_log needs a reducer: Annotated[list[str], add]"
assert ann["issue_log"].__metadata__[0] is add, \
    "issue_log's reducer should be operator.add (list concatenation)"
assert not hasattr(ann["issues"], "__metadata__"), \
    "issues must NOT have a reducer — an accumulating control field never empties, so the graph loops forever"

print("PASS — audit field accumulates, control field overwrites.")

"""
EXPECTED OUTPUT
---------------
PASS — audit field accumulates, control field overwrites.
"""

In [ ]:
# Provided — why the framework CARES about reducers (read, run, no changes needed).
# Two nodes that run in the same superstep and write the same un-reduced key is a data race.
# LangGraph refuses rather than silently picking a winner. Beginners read this error as a
# framework bug; it is the framework catching a real bug in your design.
from langgraph.graph import StateGraph, START, END
from langgraph.errors import InvalidUpdateError

class RaceState(TypedDict):
    findings: list[str]        # no reducer — deliberately

def scanner_a(state): return {"findings": ["found by A"]}
def scanner_b(state): return {"findings": ["found by B"]}

race = StateGraph(RaceState)
race.add_node("a", scanner_a)
race.add_node("b", scanner_b)
race.add_edge(START, "a")      # both start together -> both write in the SAME superstep
race.add_edge(START, "b")
race.add_edge("a", END)
race.add_edge("b", END)

try:
    race.compile().invoke({"findings": []})
    print("no error (unexpected)")
except InvalidUpdateError as e:
    print("InvalidUpdateError as expected:")
    print(" ", str(e).splitlines()[0])

# Same graph, one line different: give the channel a merge strategy.
class SafeState(TypedDict):
    findings: Annotated[list[str], add]

safe = StateGraph(SafeState)
safe.add_node("a", scanner_a)
safe.add_node("b", scanner_b)
safe.add_edge(START, "a")
safe.add_edge(START, "b")
safe.add_edge("a", END)
safe.add_edge("b", END)
print("with a reducer:", safe.compile().invoke({"findings": []}))

"""
EXPECTED OUTPUT
---------------
InvalidUpdateError as expected:
  At key 'findings': Can receive only one value per step. Use an Annotated key to handle multiple values.
with a reducer: {'findings': ['found by A', 'found by B']}
"""

---
## Part B2 · Nodes are just functions

This is the demystifying fact: **a LangGraph node is a plain Python function** with the signature `state -> partial update`. No base class, no decorator, no framework magic. You can call it directly with a dict, which means you can unit-test every node without ever building a graph.

The checks below are deliberately deterministic (string matching, not an LLM). Two reasons:

1. The lab is about orchestration, so a non-reproducible node would turn debugging into prompt-tuning.
2. **It is also the right production instinct.** If a rule can be expressed deterministically, do not pay a model to apply it inconsistently. Reserve the model for the parts that genuinely need judgement.

In [ ]:
# Provided — the policy rules and the document fixtures (read, no changes needed).
import re

REQUIRED_SECTIONS = ["Purpose", "Scope", "Effective Date"]
BANNED_TERMS      = ["guaranteed", "risk-free"]
MAX_WORDS         = 150

DRAFT_DEFECTIVE = (
    "Purpose: This policy describes remote working arrangements.\n"
    "Scope: Applies to all full-time staff.\n"
    "Employees are guaranteed approval for any remote request submitted in advance.\n"
)

DRAFT_CLEAN = (
    "Purpose: This policy describes remote working arrangements.\n"
    "Scope: Applies to all full-time staff.\n"
    "Effective Date: 2026-01-01.\n"
    "Requests should be submitted two weeks in advance through the HR portal.\n"
)

def check_document(draft: str) -> list[str]:
    """Pure function. No state, no graph, no model — trivially testable."""
    issues = []
    for section in REQUIRED_SECTIONS:
        if f"{section}:" not in draft:
            issues.append(f"missing section: {section}")
    for term in BANNED_TERMS:
        if term.lower() in draft.lower():
            issues.append(f"banned term: {term}")
    words = len(draft.split())
    if words > MAX_WORDS:
        issues.append(f"too long: {words} words > {MAX_WORDS}")
    return issues

print("defective ->", check_document(DRAFT_DEFECTIVE))
print("clean     ->", check_document(DRAFT_CLEAN))

"""
EXPECTED OUTPUT
---------------
defective -> ['missing section: Effective Date', 'banned term: guaranteed']
clean     -> []
"""

In [ ]:
# Wrap the pure check function as a NODE.
# A node returns a PARTIAL update - only the keys it changed:
#   - "issues"    : the current issue list          -> overwritten every time this node runs
#   - "issue_log" : the same issues, tagged by revision -> appended, because of the reducer
# Returning untouched keys is how people accidentally clobber another node's work, so don't.
def checks_node(state: ReviewState) -> dict:
    issues = check_document(state["draft"])
    rev = state.get("revision_count", 0)
    return {"issues": issues,
            "issue_log": [f"rev{rev}: {i}" for i in issues]}

# Call the node directly, with a plain dict. No graph needed - this is the whole point.
probe = {"doc_id": "D1", "draft": DRAFT_DEFECTIVE, "issues": [], "issue_log": [],
         "revision_count": 0, "status": "", "approver_note": ""}
print(checks_node(probe))

"""
EXPECTED OUTPUT
---------------
{'issues': ['missing section: Effective Date', 'banned term: guaranteed'], 'issue_log': ['rev0: missing section: Effective Date', 'rev0: banned term: guaranteed']}
"""

### The one node that gets a model

Everything that *decides* in this lab is deterministic on purpose. But re-read the rule from Part B2:
*reserve the model for the parts that genuinely need judgement.* Rewriting prose to satisfy a
reviewer's objections is exactly that part — it is fuzzy, and no `re.sub` will ever do it well. So
`revise` is the one node where an LLM belongs.

What makes it safe to drop a non-deterministic model inside a workflow you still trust is that
**the deterministic checker stays the judge.** The model *produces* a new draft; `check_document()` —
the same pure function your router already reads — *decides* whether it passed, and the router sends
it round the loop again if it did not. That is the habit to carry to work: **let the model produce,
let deterministic code decide.**

The cell below defines both versions of the node. `revise_node` is the deterministic one used by the
graph in the self-checks and the kernel-restart milestone, so those stay reproducible;
`llm_revise_node` is the real-model one, and the cell runs it immediately so you can watch the
produce/decide split happen.


In [ ]:
# Two implementations of the SAME node contract (state -> partial update): one deterministic,
# one backed by a real model. The graph can use either - that interchangeability is the point.

def revise_node(state: ReviewState) -> dict:
    """Deterministic repair. Used by the graph below so every self-check stays reproducible."""
    draft = state["draft"]
    for term in BANNED_TERMS:
        draft = re.sub(term, "[removed]", draft, flags=re.IGNORECASE)
    for section in REQUIRED_SECTIONS:
        if f"{section}:" not in draft:
            draft += f"{section}: TBD by document owner.\n"
    return {"draft": draft, "revision_count": state.get("revision_count", 0) + 1}

def llm_revise_node(state: ReviewState) -> dict:
    """Same contract, but a model does the rewriting. Its ONLY job is to produce a new draft;
    whether that draft is acceptable is decided downstream by check_document()."""
    issues = check_document(state["draft"])              # tell the model exactly what to fix
    rewritten = ask(
        system=("You revise short internal policy documents. Return ONLY the revised document text, "
                "no preamble. Keep every 'Key: value' line on its own line. Remove any banned "
                f"wording. Add any missing required section as its own 'Section: ...' line. "
                f"Stay under {MAX_WORDS} words."),
        user=f"DRAFT:\n{state['draft']}\n\nISSUES TO FIX:\n" + "\n".join(f"- {i}" for i in issues))
    if rewritten is None:                                # no model available -> deterministic repair
        return revise_node(state)
    return {"draft": rewritten, "revision_count": state.get("revision_count", 0) + 1}

def finalize_node(state: ReviewState) -> dict:
    """The irreversible step - publishing. Note what it does NOT do: nothing before the human gate.
    Part B8 explains why the irreversible action must live in its own node, downstream of the pause."""
    return {"status": "published"}

# --- watch produce/decide in action -------------------------------------------------------------
before = check_document(DRAFT_DEFECTIVE)
out    = llm_revise_node({"draft": DRAFT_DEFECTIVE, "revision_count": 0})
after  = check_document(out["draft"])                    # the SAME gate the router reads

print("issues BEFORE (the model was told these):", before)
print("\n--- revised draft ---\n" + out["draft"])
print("\nissues AFTER, per the deterministic checker:", after)
print("\nIf 'after' were non-empty, the router would simply send it round the revise loop again.")
print("An imperfect model is already handled by MAX_REVISIONS and the human gate - THAT is why a")
print("fuzzy step is safe inside a workflow you still trust.")

# The deterministic node is what the graph uses from here on.
fixed = revise_node(probe)
print("\ndeterministic revise_node -> revision_count:", fixed["revision_count"],
      "| issues:", check_document(fixed["draft"]))

"""
EXPECTED OUTPUT
---------------
issues BEFORE (the model was told these): ['missing section: Effective Date', 'banned term: guaranteed']

--- revised draft ---
(the rewritten policy - exact wording varies with the model, and with no key set this is the
 deterministic repair instead)

issues AFTER, per the deterministic checker: []
...
deterministic revise_node -> revision_count: 1 | issues: []
"""

---
## Part B3 · Wiring — decide in a node, route in an edge

A **conditional edge** is a function that reads state and returns the name of the next node. Keep it a **pure function of state, with no model call inside**. The decision itself is made upstream, in a node, which *writes* the decision into state; the edge only reads it.

That one habit is what makes agentic systems testable, and it generalises far beyond LangGraph:

```
LLM output  ->  state field  ->  deterministic router
 (fuzzy)         (typed)          (unit-testable)
```

Target graph for this lab:

```
START -> checks -> ◇route_after_checks◇ -> revise -> (back to checks)
                                        \-> human_approval -> ◇route_after_human◇ -> finalize -> END
                                                                                   \-> END (rejected)
```

In [ ]:
# The router, and the graph.
# SPECIFICATION for route_after_checks:
#   - if there are issues AND revision_count < MAX_REVISIONS  -> "revise"
#   - otherwise                                                -> "approval"
# (Going to approval WITH unresolved issues is intentional: after MAX_REVISIONS the machine
#  gives up and asks a human, which is exactly what a well-designed workflow should do.)
def route_after_checks(state: ReviewState) -> str:
    if state["issues"] and state.get("revision_count", 0) < MAX_REVISIONS:
        return "revise"
    return "approval"

builder = StateGraph(ReviewState)
builder.add_node("checks", checks_node)
builder.add_node("revise", revise_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "checks")

# The third argument maps the router's RETURN VALUES to node names. A mismatch here is the most
# common wiring bug: the router returns "approval" but the map says "approve", and the graph raises.
# (The map is also what the visualiser reads to label the dotted edges.)
builder.add_conditional_edges("checks", route_after_checks,
                              {"revise": "revise", "approval": "finalize"})

builder.add_edge("revise", "checks")       # the loop back
builder.add_edge("finalize", END)

graph_v1 = builder.compile()               # NO checkpointer yet - persistence comes in Part B5

seed = {"doc_id": "D1", "draft": DRAFT_DEFECTIVE, "issues": [], "issue_log": [],
        "revision_count": 0, "status": "", "approver_note": ""}
final = graph_v1.invoke(seed)
print("status         :", final["status"])
print("revision_count :", final["revision_count"])
print("issues (now)   :", final["issues"])
print("issue_log (all):", final["issue_log"])

"""
EXPECTED OUTPUT
---------------
status         : published
revision_count : 1
issues (now)   : []
issue_log (all): ['rev0: missing section: Effective Date', 'rev0: banned term: guaranteed']
"""

In [ ]:
# Self-check B3 — routing tested WITHOUT running the graph, plus both end-to-end paths.
# This is the payoff of "decide in a node, route in an edge": two lines, no graph, no tokens.
assert route_after_checks({"issues": ["x"], "revision_count": 0}) == "revise"
assert route_after_checks({"issues": [],    "revision_count": 0}) == "approval"
assert route_after_checks({"issues": ["x"], "revision_count": MAX_REVISIONS}) == "approval", \
    "After MAX_REVISIONS the machine must stop looping and escalate."

clean_run = graph_v1.invoke({**seed, "draft": DRAFT_CLEAN})
assert clean_run["revision_count"] == 0, "A clean document must not be revised."
assert clean_run["issue_log"] == [],     "A clean document must log nothing."

dirty_run = graph_v1.invoke(seed)
assert dirty_run["revision_count"] == 1, "A defective document takes exactly one revision here."
assert dirty_run["issues"] == [],        "issues must be EMPTY after a successful revision..."
assert len(dirty_run["issue_log"]) == 2, "...while issue_log keeps the audit trail."

print("PASS — router is correct in isolation, and both paths behave end to end.")
print("Note the last two asserts: that is the control/audit split doing its job.")

"""
EXPECTED OUTPUT
---------------
PASS — router is correct in isolation, and both paths behave end to end.
Note the last two asserts: that is the control/audit split doing its job.
"""

### Inspect the state after every node — do this the whole time you are developing

`TypedDict` is not validated at run time. A misspelled key does not raise; it silently creates a new
channel that nothing reads, and you lose twenty minutes wondering why a field is empty.

The cure is boring and universal: **stream the graph in `updates` mode and watch each node's return
value.** `stream_mode="updates"` shows you what each node *wrote*; `stream_mode="values"` shows the
full state after each superstep. Keep this cell around and re-run it whenever a node "does nothing."


In [ ]:
# Provided — the debugging habit, in three lines (read, run, keep).
print("=== updates mode: what each node WROTE ===")
for chunk in graph_v1.stream(seed, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"  {node_name:>10} -> {update}")

print("\n=== values mode: the FULL state after each superstep ===")
for i, snapshot in enumerate(graph_v1.stream(seed, stream_mode="values")):
    print(f"  step {i}: revision_count={snapshot.get('revision_count')} "
          f"issues={snapshot.get('issues')} status={snapshot.get('status')!r}")

print("\nIf a key you expected is missing here, you misspelled it in a node's return dict.")

"""
EXPECTED OUTPUT
---------------
=== updates mode: what each node WROTE ===
      checks -> {'issues': ['missing section: Effective Date', 'banned term: guaranteed'], 'issue_log': [...]}
      revise -> {'draft': '...', 'revision_count': 1}
      checks -> {'issues': [], 'issue_log': []}
    finalize -> {'status': 'published'}

=== values mode: the FULL state after each superstep ===
  step 0: revision_count=0 issues=[] status=''
  step 1: revision_count=0 issues=['missing section: Effective Date', 'banned term: guaranteed'] status=''
  step 2: revision_count=1 issues=[...] status=''
  step 3: revision_count=1 issues=[] status=''
  step 4: revision_count=1 issues=[] status='published'

(Exact step count can vary by patch version — what matters is that you can SEE every write.)
"""

In [ ]:
# Provided — read your own wiring back (read, run, no changes needed).
# Reading the graph BACK from the compiled object is how you catch a conditional edge that
# silently points at the wrong node.
print("nodes :", sorted(n for n in graph_v1.get_graph().nodes if not n.startswith("__")))
print()
show_graph(graph_v1, "Lab B v1 - draft -> checks -> (revise | finalize)")

"""
EXPECTED OUTPUT
---------------
nodes : ['checks', 'finalize', 'revise']

graph TD;
    __start__ --> checks;
    checks -. &nbsp;revise&nbsp; .-> revise;
    checks -. &nbsp;approval&nbsp; .-> finalize;
    revise --> checks;
    finalize --> __end__;

(Exact mermaid formatting varies by version. What matters: the dotted lines are your
 conditional edges, and they point where you think they point.)
"""

---
## Part B4 · Loops terminate because *you* made them terminate

The `revise → checks → revise` cycle is a genuine loop. Nothing in the framework guarantees it ends. LangGraph gives you a **recursion limit** as a safety net — a hard cap on supersteps — but a recursion limit is a *crash*, not a design.

Two layers, and you need both:

1. **A guard in your state** (`revision_count < MAX_REVISIONS`) — the intentional exit, which produces a sensible outcome (escalate to a human).
2. **The recursion limit** — the backstop that turns an infinite loop into an exception instead of a runaway bill.

The next cell removes the guard on purpose so you see what an unguarded loop looks like from the outside.

In [ ]:
# Provided — an unguarded loop, caught by the recursion limit (read, run, no changes needed).
from langgraph.errors import GraphRecursionError

def broken_revise(state: ReviewState) -> dict:
    # A "revise" step that does not actually fix anything — a very realistic bug when the
    # revision step is an LLM that quietly fails to apply the requested change.
    return {"revision_count": state.get("revision_count", 0) + 1}

def route_no_guard(state: ReviewState) -> str:
    return "revise" if state["issues"] else "approval"     # <- no revision_count check

b = StateGraph(ReviewState)
b.add_node("checks", checks_node)
b.add_node("revise", broken_revise)
b.add_node("finalize", finalize_node)
b.add_edge(START, "checks")
b.add_conditional_edges("checks", route_no_guard, {"revise": "revise", "approval": "finalize"})
b.add_edge("revise", "checks")
b.add_edge("finalize", END)

try:
    b.compile().invoke(seed, {"recursion_limit": 8})
    print("terminated (unexpected)")
except GraphRecursionError as e:
    print("GraphRecursionError as expected — the backstop fired:")
    print(" ", str(e).splitlines()[0][:100])
    print("\nThe guard in route_after_checks is what turns this crash into an escalation.")

"""
EXPECTED OUTPUT
---------------
GraphRecursionError as expected — the backstop fired:
  Recursion limit of 8 reached without hitting a stop condition.

The guard in route_after_checks is what turns this crash into an escalation.
"""

---
## Part B5 · Persistence: checkpointers, threads, time travel

Vocabulary, exactly:

| Term | Meaning |
|---|---|
| **Checkpointer** | A pluggable saver that writes a snapshot of state after every superstep. Wired at `compile()`. |
| **Thread** | One independent run, identified by `thread_id` in the config. Your persistent cursor: reuse it to resume, use a fresh one to start clean. |
| **Checkpoint** | One snapshot within a thread. A thread is an ordered list of checkpoints. |
| **Superstep** | One execution tick — all nodes scheduled for this tick run, updates merge, one checkpoint is written. |

> **Analogy.** The checkpointer is git for execution. `thread_id` is the branch name, each superstep is a commit, `get_state_history()` is `git log`, and resuming from an older checkpoint forks the thread rather than rewriting it.

Two savers matter today:

* `InMemorySaver` — lives in the Python process. Fine for a notebook cell. **Dies with the kernel.**
* `SqliteSaver` — writes to a file. Survives restarts. This is what Part B7 needs.

In [ ]:
# Provided — add a checkpointer and inspect the run (read, run, no changes needed).
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
graph_v2 = builder.compile(checkpointer=memory)          # same builder, now durable-ish

config = {"configurable": {"thread_id": "doc-42"}}       # the thread id IS the identity of this run
graph_v2.invoke(seed, config)

snap = graph_v2.get_state(config)
print("current status :", snap.values["status"])
print("pending nodes  :", snap.next, "  <- empty tuple means the run is complete")
print("checkpoints    :", sum(1 for _ in graph_v2.get_state_history(config)))

print("\nTime travel (newest first) — every superstep left a snapshot:")
for s in graph_v2.get_state_history(config):
    print(f"  next={str(s.next):<14} revision_count={s.values.get('revision_count')} "
          f"issues={len(s.values.get('issues', []))}")

"""
EXPECTED OUTPUT
---------------
current status : published
pending nodes  : ()   <- empty tuple means the run is complete
checkpoints    : 6

Time travel (newest first) — every superstep left a snapshot:
  next=()             revision_count=1 issues=0
  next=('finalize',)  revision_count=1 issues=0
  next=('checks',)    revision_count=1 issues=2
  next=('revise',)    revision_count=0 issues=2
  next=('checks',)    revision_count=0 issues=0
  next=('__start__',) revision_count=None issues=0   <- before any node ran

(The exact checkpoint COUNT can differ by a patch version — what matters is that every
 superstep left a snapshot, and that you can read the state at each one.)
"""

In [ ]:
# Provided — the limit of an in-memory checkpointer (read, run, no changes needed).
# We simulate a process restart the cheap way: build a NEW saver, exactly as a fresh Python
# process would. The thread id is the same. The state is gone.
fresh_memory = InMemorySaver()
graph_restarted = builder.compile(checkpointer=fresh_memory)
snap_after = graph_restarted.get_state(config)

print("state found after 'restart' :", bool(snap_after.values))
print("values                      :", snap_after.values)
print("\nInMemorySaver stores checkpoints in the process's heap. A restart is a wipe.")
print("Part B7 fixes this with SqliteSaver — and the lab objective ('resume across a kernel")
print("restart') is IMPOSSIBLE with InMemorySaver, no matter how the graph is written.")

"""
EXPECTED OUTPUT
---------------
state found after 'restart' : False
values                      : {}

InMemorySaver stores checkpoints in the process's heap. A restart is a wipe.
Part B7 fixes this with SqliteSaver — and the lab objective ('resume across a kernel
restart') is IMPOSSIBLE with InMemorySaver, no matter how the graph is written.
"""

---
## Part B6 · The human interrupt

Two mechanisms exist. Learn the modern one; recognise the older one when you read someone else's code.

| | **Dynamic — `interrupt()`** | **Static — breakpoints** |
|---|---|---|
| How | Called *inside* a node, conditionally, with a payload | `interrupt_before=["node"]` at compile time |
| Granularity | Any point in a node, data-dependent | Node boundaries only, always |
| Sends data to the human? | Yes — payload must be JSON-serialisable | No — the caller must fetch state itself |
| Use for | Approvals, clarifying questions, edits | Debugging, stepping through a graph |

**Three preconditions, all required.** Miss one and the error message will not tell you which:

1. a checkpointer wired at `compile()` — **before** the pause, not after;
2. a `thread_id` in the config;
3. a JSON-serialisable interrupt payload.

**The mechanics.** `interrupt(payload)` raises a resumable pause. The caller gets the payload back under the `__interrupt__` key. Later — a second later or three days later, in a different process — the caller re-invokes with `Command(resume=value)`, and that `value` becomes the **return value of `interrupt()`** inside the node.

**Design judgement — where to put the human.** Interrupt on: irreversible actions (send, publish, delete, pay, deploy), high blast radius, regulated decisions, and plan approval before expensive execution. Do **not** interrupt on reversible or cheap steps, and never on every model call — a graph that asks for approval constantly trains its reviewers to rubber-stamp, which manufactures false assurance and is worse than having no gate at all.

> **What `interrupt()` does *not* give you: authorization.** The primitive pauses the graph and
> accepts a value. It has no opinion about *who* sent that value. In production the resume call sits
> behind your own authn/authz — the identity of the approver, and the fact that they were entitled to
> approve *this* document, belongs in state (and in your audit log) as data you wrote deliberately.
> A workflow that records "approved" without recording "by whom, at what time, under what authority"
> will not survive its first audit.


In [ ]:
# The human approval node.
# The payload is what a human will actually look at. "Approve?" with no context guarantees
# a rubber stamp; give the reviewer the document, the outstanding issues, and how many
# repair attempts were already made.
from langgraph.types import interrupt, Command

def human_approval_node(state: ReviewState) -> dict:
    # interrupt() raises a RESUMABLE pause. The payload is handed to the caller under the
    # "__interrupt__" key; whatever the caller later sends back via Command(resume=...) becomes
    # the RETURN VALUE of this call. Everything must be JSON-serialisable, both ways.
    decision = interrupt({
        "question": "Approve this document?",
        "doc_id": state["doc_id"],
        "draft": state["draft"],
        "outstanding_issues": state["issues"],
        "revisions_attempted": state["revision_count"],
    })

    # Write the human's decision into state. Supported actions: "approved" | "rejected" | "changes_requested".
    # If the human edited the text, their version REPLACES the draft - this is the moment a human
    # authors state in the middle of a machine's run.
    return {"status": decision["action"],
            "approver_note": decision.get("note", ""),
            "draft": decision.get("edited_draft", state["draft"])}

def route_after_human(state: ReviewState) -> str:
    if state["status"] == "approved":          return "finalize"
    if state["status"] == "changes_requested": return "revise"
    return "end"                                # rejected -> stop

# Rebuild the graph with the approval gate wired between checks and finalize.
g = StateGraph(ReviewState)
g.add_node("checks", checks_node)
g.add_node("revise", revise_node)
g.add_node("human_approval", human_approval_node)
g.add_node("finalize", finalize_node)
g.add_edge(START, "checks")
g.add_conditional_edges("checks", route_after_checks,
                        {"revise": "revise", "approval": "human_approval"})
g.add_edge("revise", "checks")
g.add_conditional_edges("human_approval", route_after_human,
                        {"finalize": "finalize", "revise": "revise", "end": END})
g.add_edge("finalize", END)

approval_graph = g.compile(checkpointer=InMemorySaver())   # checkpointer BEFORE the pause
show_graph(approval_graph, "Lab B - approval graph with interrupt")
print("graph compiled with a human-approval interrupt.")

"""
EXPECTED OUTPUT
---------------
a picture with the human_approval node between checks and finalize, then:
graph compiled with a human-approval interrupt.
"""

In [ ]:
# Provided — run until the graph pauses (read, run, no changes needed).
cfg = {"configurable": {"thread_id": "doc-approve-1"}}
result = approval_graph.invoke(seed, cfg)

print("__interrupt__ present :", "__interrupt__" in result)
payload = result["__interrupt__"][0].value
print("\nWhat the human is asked:")
for k, v in payload.items():
    print(f"  {k:>20} : {v!r}" if k != "draft" else f"  {k:>20} : {v[:48]!r}...")

snap = approval_graph.get_state(cfg)
print("\npending node          :", snap.next, " <- the graph is parked here, indefinitely")

"""
EXPECTED OUTPUT
---------------
__interrupt__ present : True

What the human is asked:
              question : 'Approve this document?'
                doc_id : 'D1'
                 draft : 'Purpose: This policy describes remote working ar'...
    outstanding_issues : []
   revisions_attempted : 1

pending node          : ('human_approval',)  <- the graph is parked here, indefinitely
"""

In [ ]:
# Resume the paused graph with the human's answer.
# Note what the input to invoke() is: NOT new state, but a Command carrying the resume value.
# The config must contain the SAME thread_id, or you silently start a brand-new run.
done = approval_graph.invoke(
    Command(resume={"action": "approved", "note": "Fine to publish."}), cfg)

print("status        :", done["status"])
print("approver_note :", done["approver_note"])
print("pending       :", approval_graph.get_state(cfg).next)

"""
EXPECTED OUTPUT
---------------
status        : published
approver_note : Fine to publish.
pending       : ()
"""

In [ ]:
# Provided — the other two response shapes (read, run, no changes needed).
# One primitive, three behaviours. Each runs on its OWN thread_id: thread ids must be unique
# per task instance, or one person's approval resumes someone else's workflow.

# 2. REJECT -> the graph stops without publishing.
cfg_reject = {"configurable": {"thread_id": "doc-approve-2"}}
approval_graph.invoke(seed, cfg_reject)
rejected = approval_graph.invoke(
    Command(resume={"action": "rejected", "note": "Legal has not signed off."}), cfg_reject)
print("REJECT -> status:", rejected["status"], "| note:", rejected["approver_note"])

# 3. EDIT THEN APPROVE -> the human's text replaces the draft, mid-run.
cfg_edit = {"configurable": {"thread_id": "doc-approve-3"}}
approval_graph.invoke(seed, cfg_edit)
edited = approval_graph.invoke(
    Command(resume={"action": "approved",
                    "note": "Tightened the wording.",
                    "edited_draft": DRAFT_CLEAN + "Reviewed and edited by the policy owner.\n"}),
    cfg_edit)
print("EDIT   -> status:", edited["status"])
print("EDIT   -> last line of draft:", edited["draft"].strip().splitlines()[-1])

"""
EXPECTED OUTPUT
---------------
REJECT -> status: rejected | note: Legal has not signed off.
EDIT   -> status: published
EDIT   -> last line of draft: Reviewed and edited by the policy owner.
"""

In [ ]:
# Self-check B6 — all three human responses produce distinct, correct outcomes.
assert done["status"] == "published",     "approve must reach finalize"
assert rejected["status"] == "rejected",  "reject must NOT reach finalize"
assert "Reviewed and edited" in edited["draft"], "the human's edit must be written into state"
assert approval_graph.get_state(cfg).next == (), "an approved thread should be complete"

print("PASS — approve, reject, and edit-then-approve all behave correctly.")
print("One primitive (interrupt/resume) covers every human-in-the-loop shape you need.")

"""
EXPECTED OUTPUT
---------------
PASS — approve, reject, and edit-then-approve all behave correctly.
One primitive (interrupt/resume) covers every human-in-the-loop shape you need.
"""

> **One more path, and its limit.** `route_after_human` also supports `"changes_requested"`, which
> sends the document back through `revise → checks` and then straight back to the human. Notice what
> `MAX_REVISIONS` does and does not bound: it bounds the number of times the **machine** loops on
> its own, not the number of times a **human** can send the document back. Human-driven loops are
> unbounded by design — a reviewer is allowed to be stubborn. If your process needs a ceiling on
> that too (escalate to a second approver after three rejections), it is a second counter and a
> second guard, and you write it yourself.


---
## Part B7 · The milestone — surviving a real kernel restart

Everything so far has been in one process. The actual requirement is harsher: a human may approve
**three days later**, from a web form, after the server has been redeployed twice.

The change that makes that work is one line: a **durable checkpointer**. `SqliteSaver` writes each
checkpoint to a file instead of the heap, so the paused run outlives the process.

One thing to internalise before you run it: **a checkpointer persists your STATE, never your CODE.**
After a restart, the node functions and the wiring have to exist again before you can resume — in a
real service that is just your repo being imported at startup, and in this notebook it is the
self-contained cell you will run after the kernel restart. Nothing about the *state* is lost either
way; it is sitting in `approvals.sqlite`.


In [ ]:
# Start a durable run and PAUSE it.
import os, sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH   = "approvals.sqlite"
THREAD_ID = "doc-durable-1"

if os.path.exists(DB_PATH):        # clean slate so the lab is repeatable
    os.remove(DB_PATH)

# check_same_thread=False: the runtime may touch the connection from a worker thread.
conn  = sqlite3.connect(DB_PATH, check_same_thread=False)
saver = SqliteSaver(conn)
saver.setup()                      # creates the checkpoint tables; idempotent

# Same builder pattern as before - only the checkpointer changes. Everything else about the graph
# is identical, which is exactly the claim "durability is a deployment concern, not a design one".
durable_graph = g.compile(checkpointer=saver)
dcfg = {"configurable": {"thread_id": THREAD_ID}}

seed_doc = {"doc_id": "D-9", "draft": DRAFT_DEFECTIVE, "issues": [], "issue_log": [],
            "revision_count": 0, "status": "", "approver_note": ""}
paused = durable_graph.invoke(seed_doc, dcfg)

print("paused at        :", durable_graph.get_state(dcfg).next)
print("checkpoint file  :", DB_PATH, f"({os.path.getsize(DB_PATH)} bytes on disk)")
print("thread id        :", THREAD_ID)
print("\nThis run now exists ON DISK. Restart the kernel and it will still be there.")

"""
EXPECTED OUTPUT
---------------
paused at        : ('human_approval',)
checkpoint file  : approvals.sqlite (NNNNN bytes on disk)
thread id        : doc-durable-1

This run now exists ON DISK. Restart the kernel and it will still be there.
"""

> **Production note — checkpoint deserialization.** A checkpoint file is a deserialization surface.
> If an attacker can write to your checkpoint store, they may be able to influence what gets
> reconstructed when you resume. LangGraph's checkpoint libraries let you restrict this: set the
> environment variable `LANGGRAPH_STRICT_MSGPACK=true`, or pass an explicit allow-list of modules
> when constructing the checkpointer, so only known-safe types can be deserialized. Everything in
> *this* lab's state is a primitive (`str`, `int`, `list[str]`), so strict mode is a free win here —
> and treating your checkpoint store as trusted infrastructure, backed up and access-controlled like
> a database, is the habit to carry to work.


> ## ⛔ RESTART THE KERNEL NOW
>
> **Kernel → Restart Kernel** (do *not* run the cells above again).
>
> Every variable, every node function, every compiled graph is now gone — exactly as if your server had been redeployed while the reviewer was at lunch. The only survivor is `approvals.sqlite`.
>
> Then run the cells below. They are fully self-contained: they rebuild the *code* from scratch and reattach it to the *state* on disk. In a real service the rebuilding is just `import` — the point is that it has to happen, and that nothing about the paused run depends on it.


In [ ]:
# Run this AFTER restarting the kernel. Nothing from the cells above is in memory - so this cell
# rebuilds the graph definition from scratch (in a service: your repo, imported at startup) and
# reattaches it to the checkpoints sitting in approvals.sqlite.
import re, sqlite3
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver

MAX_REVISIONS     = 3
REQUIRED_SECTIONS = ["Purpose", "Scope", "Effective Date"]
BANNED_TERMS      = ["guaranteed", "risk-free"]
MAX_WORDS         = 150

class ReviewState(TypedDict):
    doc_id: str
    draft: str
    issues: list[str]
    revision_count: int
    status: str
    approver_note: str
    issue_log: Annotated[list[str], add]      # the schema MUST match the one the checkpoints were
                                              # written with - see the pitfall table below

def check_document(draft: str) -> list[str]:
    issues  = [f"missing section: {s}" for s in REQUIRED_SECTIONS if f"{s}:" not in draft]
    issues += [f"banned term: {t}" for t in BANNED_TERMS if t.lower() in draft.lower()]
    if len(draft.split()) > MAX_WORDS:
        issues.append(f"too long: {len(draft.split())} words > {MAX_WORDS}")
    return issues

def checks_node(state):
    rev = state.get("revision_count", 0)
    issues = check_document(state["draft"])
    return {"issues": issues, "issue_log": [f"rev{rev}: {i}" for i in issues]}

def revise_node(state):
    draft = state["draft"]
    for term in BANNED_TERMS:
        draft = re.sub(term, "[removed]", draft, flags=re.IGNORECASE)
    for section in REQUIRED_SECTIONS:
        if f"{section}:" not in draft:
            draft += f"{section}: TBD by document owner.\n"
    return {"draft": draft, "revision_count": state.get("revision_count", 0) + 1}

def human_approval_node(state):
    decision = interrupt({"question": "Approve this document?", "doc_id": state["doc_id"],
                          "draft": state["draft"], "outstanding_issues": state["issues"],
                          "revisions_attempted": state["revision_count"]})
    return {"status": decision["action"],
            "approver_note": decision.get("note", ""),
            "draft": decision.get("edited_draft", state["draft"])}

def finalize_node(state):
    return {"status": "published"}

def route_after_checks(state):
    if state["issues"] and state.get("revision_count", 0) < MAX_REVISIONS:
        return "revise"
    return "approval"

def route_after_human(state):
    if state["status"] == "approved":          return "finalize"
    if state["status"] == "changes_requested": return "revise"
    return "end"

def build_graph(checkpointer):
    """Everything above, wired. This function is the 'code' half of durable execution;
    approvals.sqlite is the 'state' half. Neither is useful without the other."""
    g = StateGraph(ReviewState)
    g.add_node("checks", checks_node)
    g.add_node("revise", revise_node)
    g.add_node("human_approval", human_approval_node)
    g.add_node("finalize", finalize_node)
    g.add_edge(START, "checks")
    g.add_conditional_edges("checks", route_after_checks,
                            {"revise": "revise", "approval": "human_approval"})
    g.add_edge("revise", "checks")
    g.add_conditional_edges("human_approval", route_after_human,
                            {"finalize": "finalize", "revise": "revise", "end": END})
    g.add_edge("finalize", END)
    return g.compile(checkpointer=checkpointer)

conn  = sqlite3.connect("approvals.sqlite", check_same_thread=False)
saver = SqliteSaver(conn)                    # reattach to the STATE
saver.setup()

graph = build_graph(saver)                   # rebuild the CODE
cfg   = {"configurable": {"thread_id": "doc-durable-1"}}    # same thread id = same run

snap = graph.get_state(cfg)
print("recovered from disk :", bool(snap.values))
print("parked at           :", snap.next)
print("doc_id              :", snap.values.get("doc_id"))

# The human finally answers - in a brand-new process.
final = graph.invoke(Command(resume={"action": "approved", "note": "Approved after restart."}), cfg)
print("\nstatus              :", final["status"])
print("approver_note       :", final["approver_note"])
print("audit trail         :", final["issue_log"])
print("\nMILESTONE: paused in one process, resumed in another. That is durable execution.")

"""
EXPECTED OUTPUT
---------------
recovered from disk : True
parked at           : ('human_approval',)
doc_id              : D-9

status              : published
approver_note       : Approved after restart.
audit trail         : ['rev0: missing section: Effective Date', 'rev0: banned term: guaranteed']

MILESTONE: paused in one process, resumed in another. That is durable execution.
"""

### The finished graph

Read the picture before you read the checklist. Four things should be visible in it, and each one is
a design decision you made rather than a default you accepted:

* **Dotted edges are conditional** — they are your two routers, and their labels are the strings
  those routers return. If a label points somewhere you did not intend, that is a wiring bug you can
  see rather than debug.
* **`checks → revise → checks` is a cycle** — the revision loop, terminated by `MAX_REVISIONS` and
  nothing else.
* **`human_approval` sits between the checks and the publish**, which is where the process can park
  for three days.
* **`finalize` is its own node**, downstream of the pause — the reason why is Part B8, immediately
  below.


In [ ]:
# The graph you just resumed, drawn. (Self-contained: uses only what this post-restart cell built.)
from IPython.display import Image, display, Markdown

def show_graph(app, title: str = ""):
    if title:
        print(title); print("-" * len(title))
    try:
        display(Image(app.get_graph().draw_mermaid_png()))     # rendered by mermaid.ink
    except Exception as e:
        print(f"(mermaid.ink unavailable - {type(e).__name__}; Mermaid source below)")
        display(Markdown("```mermaid\n" + app.get_graph().draw_mermaid() + "\n```"))

print("nodes :", sorted(n for n in graph.get_graph().nodes if not n.startswith("__")))
print()
show_graph(graph, "Lab B - the finished document approval workflow")

"""
EXPECTED OUTPUT
---------------
nodes : ['checks', 'finalize', 'human_approval', 'revise']

+ the graph picture: __start__ -> checks, dotted edges checks->revise / checks->human_approval,
  revise -> checks (the loop), dotted edges human_approval -> finalize / revise / __end__,
  finalize -> __end__.
(If mermaid.ink is unreachable from your network you get the Mermaid source instead - same content.)
"""

---
## Part B8 · The trap that reaches production

**On resume, the node re-runs from the top.**

LangGraph does not restore a Python call stack. It replays the node, and when execution reaches `interrupt()` again, that call returns the resume value instead of pausing. The consequence is the pitfall that survives to production:

> **Any side effect placed before `interrupt()` in the same node executes twice.** The email sends twice. The counter double-increments. The payment API is called twice.

Two rules follow, and they are the most valuable thing in this notebook:

1. A node containing `interrupt()` should do **nothing** before it except read state.
2. Every irreversible action belongs in its **own node, downstream of the pause** — which is exactly why `finalize` is a separate node in your graph rather than the tail of `human_approval`.

(Related: call `interrupt()` at most once per node invocation. If you need several answers, use several nodes, or loop back through a conditional edge.)

In [ ]:
# Provided — watch the double execution, then watch the fix (read, run, no changes needed).
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class S(TypedDict):
    sent: int

emails_sent = []          # stands in for anything irreversible: an API call, a payment, a deploy

# --- BROKEN: side effect sits before the pause, inside the same node ---
def notify_then_ask(state: S) -> dict:
    emails_sent.append("notification")        # <-- runs on the first pass AND on the resume
    approved = interrupt("Approve?")
    return {"sent": len(emails_sent)}

bad = StateGraph(S)
bad.add_node("notify_then_ask", notify_then_ask)
bad.add_edge(START, "notify_then_ask")
bad.add_edge("notify_then_ask", END)
bad_graph = bad.compile(checkpointer=InMemorySaver())

c = {"configurable": {"thread_id": "trap"}}
bad_graph.invoke({"sent": 0}, c)
print("emails after the PAUSE  :", len(emails_sent))
bad_graph.invoke(Command(resume=True), c)
print("emails after the RESUME :", len(emails_sent), " <-- sent twice, and nobody logged an error")

# --- FIXED: the pause node only asks; the side effect lives downstream ---
emails_sent.clear()

def ask(state: S) -> dict:
    approved = interrupt("Approve?")          # nothing before it but reading state
    return {"sent": 1 if approved else 0}

def notify(state: S) -> dict:
    if state["sent"]:
        emails_sent.append("notification")    # runs once, after the pause is resolved
    return {}

good = StateGraph(S)
good.add_node("ask", ask)
good.add_node("notify", notify)
good.add_edge(START, "ask")
good.add_edge("ask", "notify")
good.add_edge("notify", END)
good_graph = good.compile(checkpointer=InMemorySaver())

c2 = {"configurable": {"thread_id": "fixed"}}
good_graph.invoke({"sent": 0}, c2)
print("\nfixed: emails after the PAUSE  :", len(emails_sent))
good_graph.invoke(Command(resume=True), c2)
print("fixed: emails after the RESUME :", len(emails_sent), " <-- exactly once")

"""
EXPECTED OUTPUT
---------------
emails after the PAUSE  : 1
emails after the RESUME : 2  <-- sent twice, and nobody logged an error

fixed: emails after the PAUSE  : 0
fixed: emails after the RESUME : 1  <-- exactly once
"""

In [ ]:
# Final self-check - the Milestone 5 checklist for your capstone.
import os

checklist = {
    "typed state schema with an explicit reducer":
        hasattr(ReviewState.__annotations__["issue_log"], "__metadata__"),
    "deterministic checks separated from routing":
        callable(check_document),
    "conditional routing as a pure function":
        route_after_checks({"issues": ["x"], "revision_count": 0}) == "revise",
    "loop guard (bounded revisions)":
        route_after_checks({"issues": ["x"], "revision_count": 99}) == "approval",
    "human gate on the irreversible step":
        "human_approval" in graph.get_graph().nodes,
    "durable checkpointer (survives restart)":
        os.path.exists("approvals.sqlite"),
    "irreversible action isolated downstream of pause":
        "finalize" in graph.get_graph().nodes,
}
for item, ok in checklist.items():
    print(f"  [{'x' if ok else ' '}] {item}")
assert all(checklist.values()), "One or more milestone criteria are not met."
print("\nPASS - this graph is a valid Milestone 5 skeleton.")

"""
EXPECTED OUTPUT
---------------
  [x] typed state schema with an explicit reducer
  [x] deterministic checks separated from routing
  [x] conditional routing as a pure function
  [x] loop guard (bounded revisions)
  [x] human gate on the irreversible step
  [x] durable checkpointer (survives restart)
  [x] irreversible action isolated downstream of pause

PASS - this graph is a valid Milestone 5 skeleton.
"""

---
### What you should be able to do now

- Place a problem on the autonomy spectrum and name the pattern — including which of ReAct,
  Planner-Executor, Reflection and Supervisor-Worker fits, and what each one costs.
- Design a state schema and justify, per field, whether it overwrites or accumulates.
- Write nodes as testable pure-ish functions and route with logic you can test without a graph.
- Bound a loop deliberately, rather than discovering the recursion limit in production.
- Pause a workflow for a human, resume it in a different process, and support approve / reject / edit.
- Name the one thing that must never sit before an `interrupt()`.

### Pitfall table (keep this)

| Symptom | Cause | Fix |
|---|---|---|
| `InvalidUpdateError` on a key | Two nodes wrote it in one superstep, no reducer | `Annotated[list, add]`, or give one node ownership |
| State field mysteriously empty | Node returned the whole state, or a key was misspelled — `TypedDict` does not validate at run time | Print the returned dict from every node while developing |
| "No checkpointer" on interrupt | Checkpointer not wired at `compile()` | Wire it *before* the pause, not after |
| Resume starts a fresh run | Different or missing `thread_id` | Reuse the exact config dict; print the thread id |
| Resume fails after restart | `InMemorySaver` | `SqliteSaver` / Postgres saver |
| Something happened twice | Side effect before `interrupt()` in the same node | Move it to a downstream node |
| Old thread will not resume | State schema changed after checkpoints were written | New `thread_id`, or delete the sqlite file — freeze the schema early |
| Graph never terminates | Loop with no guard, or an accumulating control field | Guard in state + recursion limit; split control from audit fields |
| `AuthenticationError` / `RateLimitError` from a node | Key missing or invalid in `.env`, or `load_dotenv()` never ran | Re-run the setup cell — it probes the model once and prints `LLM_ENABLED` |
| Agent loops on the same tool forever | No step cap; the model never stops asking | Cap tool steps in the conditional edge, as in the ReAct cell |

### Stretch goals (optional)

1. **Escalation by risk.** Add a `risk` field. Interrupt only when `risk == "high"`; auto-approve low-risk documents. This is the discipline of *rationing* human attention rather than spending it uniformly.
2. **Time travel.** Use `get_state_history()` to find the checkpoint just before the human approved, resume from that older checkpoint with a different decision, and compare the two forked threads.
3. **Approval TTL.** A human may never answer. Query the checkpointer for threads older than your TTL and auto-reject them. Every production HITL system needs this and almost no tutorial mentions it.
4. **Let the model do the revising for real.** Swap `revise_node` for `llm_revise_node` in `build_graph`, re-run the self-checks, and notice which assertions you can no longer make. That difficulty is the cost Lab A told you that you would pay — and the deterministic checker is what keeps it affordable.
5. **Add a gate to an agent.** Take the ReAct graph from Part A3 and put an `interrupt()` in front of one tool, so a human approves that tool call before it runs. That single edit is the difference between a demo agent and one you would point at a production system.


---
## Closing — the two labs are one argument

Lab A gave you a procedure for **refusing** autonomy you do not need, and the four patterns worth
reaching for when you do. Lab B showed you what you get in exchange for that refusal: a graph whose
routing you can unit-test in two lines, whose loops terminate because you bounded them, and whose
state survives a redeploy.

Carry four sentences to your capstone:

1. **Autonomy is a cost you pay for irreducible uncertainty**, not a feature you buy for its own sake.
2. **Decide in a node, route in an edge** — that is what keeps a fuzzy model output behind a testable boundary.
3. **Let the model produce; let deterministic code decide** — every pattern in Part A3 is bounded by something the model does not control.
4. **The irreversible step lives in its own node, downstream of every pause** — because on resume, the node re-runs from the top.

**Milestone 5** for every capstone problem statement is "orchestrated LangGraph workflow with
checkpointing." The graph you rebuilt in the post-restart cell is a valid skeleton for it: swap the
document policy for your domain's checks, swap `SqliteSaver` for Postgres when you deploy, and keep
the human gate exactly where it is.
